# Fast MF using eALS

This notebook is an implementation of eALS (element-wise ALS) for implicit feedback as described in *Fast Matrix Factorization for Online Recommendation with Implicit Feedback* (He et al., SIGIR 2016).

## Summary

1. Data preparation from raw review text (we chose the Amazon movies dataset).
2. eALS training with coordinate updates.
3. Offline ranking evaluation (HR@K, NDCG@K).
4. Saving artifacts for analysis/reuse.

This project separates logic into Python modules (`src/*`) and uses the notebook as an orchestrator.

In [1]:
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from src.data_ingest import prepare_data
from src.evaluate import evaluate_model
from src.train_eals import train_eals

REPO = Path(os.getcwd()).resolve()

DATA_DIR = REPO / "data"

DATASET = DATA_DIR / "AmazonMoviesDataset.txt"
if not DATASET.exists():
    raise FileNotFoundError(f"Expected dataset at: {DATASET}")

## Step 0. Choose Runtime Profile

There are two runtime modes:
- `smoke`: fast feedback loop for development and debugging, uses only a subset of the data.
- `full`: paper-aligned scale/configuration for real experiments.

In [2]:
MODE = "full"
MAX_SMOKE_RECORDS = 10000
if MODE.lower() not in {"full", "smoke"}:
    raise ValueError(f"Unsupported mode: {MODE}")

DATASET_SMOKE = DATA_DIR / "AmazonMoviesSmokeSubset.txt"

OUTPUT_DIR = REPO / "outputs" / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "smoke" and not DATASET_SMOKE.exists():
    records = 0
    with (
        DATASET.open("r", encoding="utf-8", errors="ignore") as fin,
        DATASET_SMOKE.open("w", encoding="utf-8") as fout,
    ):
        block = []
        for line in fin:
            if line.strip() == "":
                if block:
                    fout.writelines(block)
                    fout.write("\n")
                    records += 1
                    block = []
                    if records >= MAX_SMOKE_RECORDS:
                        break
                continue
            block.append(line)
        if records < MAX_SMOKE_RECORDS and block:
            fout.writelines(block)
            fout.write("\n")
            records += 1
    print(f"Created smoke subset with {records} records at {DATASET_SMOKE}")

DATASET_PATH = DATASET_SMOKE if MODE == "smoke" else DATASET
print("Dataset:", DATASET_PATH)
print("Output:", OUTPUT_DIR)

Dataset: /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/data/AmazonMoviesDataset.txt
Output: /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/outputs/full


## Step 1. Build eALS + Runtime Configs

This cell creates:
- `eals_config`: model/objective hyperparameters.
- `runtime_config`: Spark/runtime controls.

### Config from paper
- `factors=128`, `reg_lambda=0.01`, `c0=64`, `alpha=0.5`, `observed_weight=1`, `observed_value=1`, `iterations=100`, `topk=100`.

In [3]:
eals_config = {
    "factors": 128,
    "reg_lambda": 0.01,
    "c0": 64.0,
    "alpha": 0.5,
    "observed_weight": 1.0,
    "observed_value": 1.0,
    "iterations": 100,
    "topk": 100,
    "min_user_interactions": 10,
    "min_item_interactions": 10,
    "init_mean": 0.0,
    "init_std": 0.01,
    "random_seed": 42,
    "dtype": "float32",
}

runtime_config = {
    "mode": MODE,
    "num_partitions": None,
    "storage_level": "MEMORY_AND_DISK",
    "eval_user_sample": None,
    "checkpoint_dir": None,
    "spark_log_level": "WARN",
}

print("eALS config:")
print(json.dumps(eals_config, indent=2))
print("\nRuntime config:")
print(json.dumps(runtime_config, indent=2))

eALS config:
{
  "factors": 128,
  "reg_lambda": 0.01,
  "c0": 64.0,
  "alpha": 0.5,
  "observed_weight": 1.0,
  "observed_value": 1.0,
  "iterations": 100,
  "topk": 100,
  "min_user_interactions": 10,
  "min_item_interactions": 10,
  "init_mean": 0.0,
  "init_std": 0.01,
  "random_seed": 42,
  "dtype": "float32"
}

Runtime config:
{
  "mode": "full",
  "num_partitions": null,
  "storage_level": "MEMORY_AND_DISK",
  "eval_user_sample": null,
  "checkpoint_dir": null,
  "spark_log_level": "WARN"
}


## Step 2. Start Spark Session

The training/data prep pipeline is implemented with PySpark RDDs.
- Interactions and histories can be distributed.
- Factor matrices are broadcast for update passes (map-and-broadcast alternating updates).

In [4]:
spark = (
    SparkSession.builder.appName(f"eALS_notebook_{MODE}")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel(runtime_config["spark_log_level"])
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/22 23:20:33 WARN Utils: Your hostname, Felix-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.131 instead (on interface en0)
26/03/22 23:20:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 23:20:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Default parallelism: 10


## Step 3. Prepare Data

This cell preprocesses the data.

1. Parse raw review blocks into `(user, item, timestamp)`.
2. Deduplicate repeated `(user, item)` pairs (keep latest timestamp).
3. Iterative user/item minimum-count filtering (k-core style).
4. Build contiguous integer indices for users/items.
5. Split by **chronological leave-one-out** per user (latest interaction as test).
6. Compute item-specific missing-data weights \(c_i\) using popularity (paper Eq. 8).
7. Build user/item histories needed by eALS updates.

In [6]:
prep_start = time.time()
prepared = prepare_data(
    sc=sc,
    dataset_path=str(DATASET_PATH),
    eals_config=eals_config,
    runtime_config=runtime_config,
    verbose=True,
)
prep_wall = time.time() - prep_start

if prepared.num_test_users == 0:
    raise RuntimeError(
        "No test users were produced after preprocessing. "
        "Increase smoke data size or relax min_user_interactions/min_item_interactions."
    )

print(f"Preparation wall time: {prep_wall:.2f} s")
stats_df = (
    pd.DataFrame(
        [{"stage": key, "value": value} for key, value in prepared.stats.items()]
    )
    .sort_values("stage")
    .reset_index(drop=True)
)
stats_df

[prepare] loaded interactions: 7850072


[prepare] deduped interactions: 6770947


26/03/23 00:33:52 WARN ShuffleDependency: The number of shuffle blocks (4498653184) for shuffleId 30 for PairwiseRDD[219] at join at /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/src/data_ingest.py:179 with 67072 partitions is possibly too large, which could cause the driver to crash with an out-of-memory error. Consider decreasing the number of partitions in this shuffle stage.
26/03/23 00:33:52 WARN ShuffleDependency: The number of shuffle blocks (1124663296) for shuffleId 31 for PairwiseRDD[212] at reduceByKey at /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/src/data_ingest.py:172 with 33536 partitions is possibly too large, which could cause the driver to crash with an out-of-memory error. Consider decreasing the number of partitions in this shuffle stage.
26/03/23 00:33:52 WARN ShuffleDependency: The number of shuffle blocks (1124663296) for shuffleId 32 for PairwiseRDD[207] at join at /Users/Administrateur/School/X/Databas

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 53148 in stage 106.0 failed 1 times, most recent failure: Lost task 53148.0 in stage 106.0 (TID 310555) (192.168.1.131 executor driver): java.io.FileNotFoundException: /private/var/folders/f4/y03jc2rs55q9y24pqptp5dg40000gn/T/blockmgr-44344705-5d6e-4f55-b8ef-0383688581df/0c/shuffle_30_310555_0.index.a823c53b-5119-417a-8509-e1de86a9eb41 (No space left on device)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:289)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:230)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:179)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.writeMetadataFile(IndexShuffleBlockResolver.scala:506)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.writeMetadataFile

26/03/23 00:52:42 WARN Utils: Suppressing exception in finally: No space left on device
java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:367)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:125)
	at java.base/java.io.BufferedOutputStream.implFlush(BufferedOutputStream.java:252)
	at java.base/java.io.BufferedOutputStream.flush(BufferedOutputStream.java:240)
	at java.base/java.io.FilterOutputStream.close(FilterOutputStream.java:184)
	at java.base/java.io.FilterOutputStream.close(FilterOutputStream.java:193)
	at org.apache.spark.shuffle.IndexShuffleBlockResolver.$anonfun$writeMetadataFile$3(IndexShuffleBlockResolver.scala:512)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally(SparkErrorUtils.scala:95)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally$(SparkErrorUtils.scala:83)
	at org.apache.spark.util.Utils$

### See Data Structures

These are the sparse structures used to avoid iterating over the full user-item matrix.
- `user_history_rdd`: for each user, which items were observed and with what \(c_i\).
- `item_history_rdd`: for each item, which users observed it plus that item’s \(c_i\).

In [ ]:
print("num_users:", prepared.num_users)
print("num_items:", prepared.num_items)
print("num_train_interactions:", prepared.num_train_interactions)
print("num_test_users:", prepared.num_test_users)

print("\nSample user history rows:")
for row in prepared.user_history_rdd.take(3):
    u, hist = row
    print("user", u, "history_len", len(hist), "first", hist[:3])

print("\nSample item history rows:")
for row in prepared.item_history_rdd.take(3):
    i, c_i, users = row
    print(
        "item",
        i,
        "c_i",
        round(float(c_i), 6),
        "users_len",
        len(users),
        "first",
        users[:5],
    )

## Step 3. Train eALS

Training alternates between:
- updating all user vectors `P` with item cache \(S^q\) (eq. 12 from the paper),
- updating all item vectors `Q` with user cache \(S^p\) (eq. 13 from the paper).
- Cache matrices implement the fast eALS idea that reduces repeated computation over missing entries.

The output `train_log` reports per-iteration times to see progress.

In [ ]:
train_start = time.time()
model = train_eals(sc=sc, prepared_data=prepared, config=eals_config)
train_wall = time.time() - train_start
print(f"Training wall time: {train_wall:.2f} s")

train_log_df = pd.DataFrame(model.train_log)
train_log_df

## Step 4. Evaluate Ranking Quality

We evaluate recommendation quality with:
- `HR@K`: did the true held-out item appear in top-K?
- `NDCG@K`: was it ranked high in top-K?

With `K=100`, like in the paper.

In [ ]:
eval_start = time.time()
metrics = evaluate_model(
    sc=sc,
    test_rdd=prepared.test_rdd,
    p_matrix=model.p_matrix,
    q_matrix=model.q_matrix,
    topk=eals_config["topk"],
    eval_user_sample=runtime_config["eval_user_sample"],
    random_seed=eals_config["random_seed"],
)
eval_wall = time.time() - eval_start
print(f"Evaluation wall time: {eval_wall:.2f} s")
metrics

## Step 5. Save Artifacts

We save everything needed for reproducibility and analysis:
- `P.npy`, `Q.npy`: learned latent factors.
- `metrics.json`: HR/NDCG summary.
- `prepare_stats.json`: preprocessing counters/timings.
- `train_log.json`: iteration timing.
- `id_maps.json`: mapping from internal indices to raw IDs.

Keeping these files explicit makes it easy to compare runs and audit behavior.
        


In [ ]:
np.save(OUTPUT_DIR / "P.npy", model.p_matrix)
np.save(OUTPUT_DIR / "Q.npy", model.q_matrix)
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "prepare_stats.json").write_text(
    json.dumps(prepared.stats, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "train_log.json").write_text(
    json.dumps({"iterations": model.train_log}, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "id_maps.json").write_text(
    json.dumps(
        {"user_ids": prepared.user_ids, "item_ids": prepared.item_ids}, indent=2
    ),
    encoding="utf-8",
)
print("Saved artifacts to", OUTPUT_DIR)

## Step 6. Cleanup

Unpersist cached RDDs and stop Spark.

In [ ]:
prepared.train_ui_rdd.unpersist()
prepared.test_rdd.unpersist()
prepared.user_history_rdd.unpersist()
prepared.item_history_rdd.unpersist()

spark.stop()
print("Spark stopped.")